# 🏠🏠🏠 Projet Kaggle : Catboost : Premières impressions 🏠🏠🏠

## Initialisation

### Importation des bibliothèques nécessaires


In [1]:
import json
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from catboost import CatBoostRegressor, Pool
from shapash import SmartExplainer
from sklearn.metrics import (
    max_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

### Importation des données


In [2]:
with open("../data/processed/dtype_dict.json") as f:
    dtype_dict = json.load(f)

train = pd.read_csv(
    "../data/processed/train.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

test = pd.read_csv(
    "../data/processed/test.csv",
    delimiter=",",
    encoding="utf-8",
    index_col="Id",
    dtype=dtype_dict,
)

dfs = [train, test]

### Reprise des transformations intéressantes


In [3]:
neighborhoods_to_keep = [
    "Brookside",
    "Clear Creek",
    "Crawford",
    "Northridge",
    "Northridge Heights",
    "Stone Brook",
    "Veenker",
]

for df in dfs:
    df["Neighborhood_agg2"] = np.where(
        df["Neighborhood"].isin(neighborhoods_to_keep), df["Neighborhood"], "Autre"
    )

    df["FullBath_tot"] = df["FullBath"] + df["BsmtFullBath"]
    df["HalfBath_tot"] = df["HalfBath"] + df["BsmtHalfBath"]

    # Rajout d'un traitement pour Alley
    df["Alley"] = df["Alley"].fillna("No Alley")

## Premier modèle avec toutes les variables d'origines et premières impressions

### Filtrage du dataframe


In [4]:
col_suppr = []

for col in train.columns:
    if re.search(r"_(ord|agg(|2)|optb|tot|)$", col):
        col_suppr.append(col)

train_1 = train.drop(columns=col_suppr, axis=1).copy()

### Colonnes et index des variables categorielles


In [5]:
# Colonnes de type catégoriel
categorical_columns = (
    train_1.drop(columns="SalePrice", axis=1)
    .select_dtypes(include=["category", "object"])
    .columns
).to_list()

### Séparation en train test


In [6]:
df_train, df_test = train_test_split(train_1, test_size=0.25, random_state=42)

### Création des Pools


In [7]:
# Création des objets Pool pour CatBoost
train_pool = Pool(
    df_train.drop(columns="SalePrice", axis=1),
    label=df_train["SalePrice"],
    cat_features=categorical_columns,
)
test_pool = Pool(
    df_test.drop(columns="SalePrice", axis=1),
    label=df_test["SalePrice"],
    cat_features=categorical_columns,
)

### Modèle Catboost


In [8]:
# Initialiser et entraîner le modèle CatBoostClassifier
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.035,
    depth=8,
    loss_function="RMSE",
    verbose=100,
    use_best_model=True,
)

### Entrainement du modèle


In [9]:
model.fit(train_pool, eval_set=test_pool, plot=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 76389.1050371	test: 82366.4176879	best: 82366.4176879 (0)	total: 1.35s	remaining: 11m 14s
100:	learn: 23998.7261416	test: 31370.3161136	best: 31370.3161136 (100)	total: 1m 5s	remaining: 4m 17s
200:	learn: 17006.7872368	test: 27658.4326298	best: 27658.4326298 (200)	total: 1m 48s	remaining: 2m 41s
300:	learn: 13544.2815239	test: 26456.0277534	best: 26448.9927056 (297)	total: 2m 29s	remaining: 1m 38s
400:	learn: 11185.9724945	test: 26034.5596631	best: 26026.3777366 (397)	total: 3m 2s	remaining: 45s
499:	learn: 9552.1353754	test: 25923.8096749	best: 25923.8096749 (499)	total: 3m 59s	remaining: 0us

bestTest = 25923.80967
bestIteration = 499



## Analyses des performances

### Définition des différents thèmes et figures plotly


In [10]:
# Template personnalisé
monTheme = go.layout.Template(
    layout=dict(
        template="simple_white",
        autosize=True,
        font=dict(family="Arial", size=15, color="#000000"),
        title=dict(font=dict(size=35, family="Arial"), x=0.5),
        xaxis=dict(tickangle=-35, automargin=True),
        yaxis=dict(tickangle=-35, automargin=True),
    )
)

# Enregistrement du template
pio.templates["monTheme"] = monTheme

# Définition du template comme template par défaut
pio.templates.default = "monTheme"

# Même principe, style de boutons par defaut
# Ne peut pas rentrer dans les templates
styleBoutons = dict(
    bgcolor="#6B6B6B",
    bordercolor="#000000",
    borderwidth=1.5,
    direction="right",
    font_weight=700,
    showactive=True,
    type="buttons",
    x=1,
    xanchor="right",
    y=1.2,
    yanchor="top",
)

mesPolices = {
    "font-size": 25,
    "font-family": "Arial",
    "font-weight": 700,
    "color": "Black",
}

rouge = "rgb(200, 10, 10)"

res = "Résidus"

In [11]:
def plot_perf(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "MAE": [mean_absolute_error(y_true, y_pred)],
            "RMSE": [root_mean_squared_error(y_true, y_pred)],
            "MEE": [median_absolute_error(y_true, y_pred)],
            "ME": [max_error(y_true, y_pred)],
            "R2": [r2_score(y_true, y_pred)],
        }
    )

In [12]:
def residuals_density(df: pd.DataFrame, res_col: str):
    residuals_density = px.histogram(
        df, x=res_col, marginal="box", color_discrete_sequence=[rouge]
    )

    residuals_density.update_layout(
        title_text="Répartition des résidus",
        xaxis_title=res,
        yaxis_title="Nombre",
        showlegend=False,
    )

    residuals_density.show()

In [13]:
def scat_res_price(df: pd.DataFrame, res_col: str, target_col: str):
    scat_res_price = px.scatter(
        df,
        x=target_col,
        y=res_col,
        color_discrete_sequence=[rouge],
    )

    scat_res_price.update_layout(
        title_text="Résidus (valeur réelle - valeur prédite) en fonction du Prix des maisons",
        xaxis_title="Prix de vente",
        yaxis_title=res,
    )

    scat_res_price.show()

In [14]:
def scat(df, first_col: str, res_col: str, numerical_cols: list):
    # Définition de la figure type nuages de points
    scat = go.Figure(
        go.Scatter(
            x=df[first_col],
            y=df[res_col],
            mode="markers",
            marker_color=rouge,
        )
    )

    boutons_x = [
        dict(
            label=f"x - {x}",
            method="update",
            args=[
                {"x": [df[x]]},
                {"xaxis": {"title": x}},
            ],
        )
        for x in numerical_cols
    ]

    # Mise à jour du layout
    scat.update_layout(
        title_text="Relation entre les résidus et la variable quantitative séléctionnée",
        xaxis_title=first_col,
        yaxis_title=res,
        updatemenus=[
            dict(
                buttons=boutons_x,
                direction="up",  # Set to 'down' or 'up' for dropdown
                showactive=True,
                x=1,
                xanchor="right",
                y=-0.25,
                yanchor="bottom",  # Custom styles specified here
                bgcolor=styleBoutons["bgcolor"],
                bordercolor=styleBoutons["bordercolor"],
                borderwidth=styleBoutons["borderwidth"],
            ),
        ],
    )

    # Affichage de la figure
    scat.show()

In [15]:
def violin(df, first_col: str, res_col: str, categorical_cols: list):
    violin = go.Figure(
        go.Violin(
            x=df[first_col],
            y=df[res_col],
            fillcolor=rouge,
            line_color="black",
            marker_color="black",
            box_visible=True,
            meanline_visible=True,
        )
    )

    boutons_y = [
        dict(
            label=f"x - {x}",
            method="update",
            args=[
                {"x": [df[x]]},
                {"xaxis": {"title": x}},
            ],
        )
        for x in categorical_cols
    ]

    # Mise à jour du layout
    violin.update_layout(
        title_text="Relation entre les résidus et la variable qualitative sélectionnée",
        xaxis_title=first_col,
        yaxis_title=res,
        updatemenus=[
            dict(
                buttons=boutons_y,
                direction="up",  # Set to 'down' or 'up' for dropdown
                showactive=True,
                x=1,
                xanchor="right",
                y=-0.25,
                yanchor="bottom",  # Custom styles specified here
                bgcolor=styleBoutons["bgcolor"],
                bordercolor=styleBoutons["bordercolor"],
                borderwidth=styleBoutons["borderwidth"],
            ),
        ],
    )

    # Affichage de la figure
    violin.show()

### Calculs des prédictions


In [16]:
df_test["SalePrice_pred"] = model.predict(test_pool)

### Quelques métriques bien connues


In [17]:
plot_perf(df_test["SalePrice"], df_test["SalePrice_pred"])

,MAE,RMSE,MEE,ME,R2
0,15549.299811,25923.809675,10497.530686,236745.283197,0.904066


### Forme des résidus


In [18]:
df_test["residus"] = df_test["SalePrice"] - df_test["SalePrice_pred"]
residuals_density(df_test, "residus")

### Résidus en fonction du Prix de vente


In [19]:
scat_res_price(
    df_test,
    "residus",
    "SalePrice",
)

### Résidus en fonction des variables quantitatives


In [20]:
numerical_cols = [
    col
    for col in df_test.columns
    if pd.api.types.is_any_real_numeric_dtype(df_test[col])
]

scat(
    df_test,
    "LotFrontage",
    "residus",
    numerical_cols,
)

### Résidus en fonction des variables qualitatives sélectionnées


In [21]:
categorical_cols = [col for col in df_test.columns if df_test[col].dtype == "object"]

violin(df_test, "MSSubClass", "residus", categorical_cols)

Nous constatons actuellement des performances similaires à celles des modèles segmentés, avec toujours le même problème : les maisons de luxe sont sous-estimées. La question de savoir si le problème vient du modèle ou de l'estimation initiale semble légitime. Pour y répondre, il serait idéal de consulter la personne ayant mis en ligne cette base de données. La meilleure solution serait de demander l'avis d'experts immobiliers, tels que des agents ayant une connaissance approfondie du marché, de l'offre et de la demande.

### Explicabilité avec la webapp Shapash

Pour les régressions, l'interprétation peut se faire de manière assez traditionnelle. Pour les modèles ensemblistes, on parle souvent d'importance des caractéristiques, mais l'interprétation est moins évidente. Une piste sérieuse pour pallier ce manque d'explication consiste à utiliser les valeurs de Shapley.

## Quelques mots sur les valeurs de Shapley

Issues de la théorie des jeux, elles permettent d'attribuer de manière équitable la contribution de chaque variable dans un modèle. L'ère du machine learning a introduit des algorithmes complexes à interpréter, et ce n'est que récemment que ces métriques ont vu le jour. L'intérêt réside dans la capacité à interpréter, d'un point de vue global, comment une variable influence la prédiction (en fonction des modalités ou de la valeur numérique), mais aussi d'un point de vue local (à l'échelle individuelle). L'objectif est d'expliquer la prédiction du prix d'une maison associée à une observation, c'est-à-dire à l'ensemble des caractéristiques du bien (les valeurs de nos variables explicatives). Nous disposons de la moyenne de toutes les prédictions. L'idée est d'examiner l'écart entre la moyenne des prédictions et la prédiction pour une observation particulière. Cet écart est ensuite réparti et expliqué par chacune des modalités prises par la maison.

<div style="text-align: center;">
  <img src="../src/img/shapley_value.jpg" alt="Shapley Value" />
</div>

Par exemple, la maison ci-dessus peut être décrite de la manière suivante :
La prédiction est de 242 547$, la moyenne de toutes les prédictions étant de 182 711$. Plusieurs modalités jouent positivement sur le prix, comme la superficie du sous-sol, par exemple. Pour obtenir l’augmentation ou la diminution d'une modalité en particulier, il s’agit de raisonner par coalition. À toute chose égale par ailleurs, les +18 693 $ de la surface du sous-sol sont obtenus en comparant la moyenne des prédictions des maisons ayant une qualité générale de 7, une surface de garage de 650 SF, une taille de terrain de 7 050 SF, une qualité des matériaux extérieurs de 3, etc., avec des tailles de sous-sols différentes de celle de la maison en question. Dès lors, il est possible de déterminer si cette modalité joue positivement ou négativement sur la prédiction et de le quantifier. Le principe est le même pour les autres variables, les autres individus et même pour les sous-populations. C’est ce qui est intéressant avec les valeurs de Shapley : il est possible de raisonner par groupe, avec une maille plus ou moins fine, grâce au système de coalition et d’espérance conditionnelle.

Pour aller plus loin, voici quelques liens :

- [Quantemetry - Valeurs de Shapley](https://www.quantmetry.com/blog/valeurs-de-shapley/)
- [Statistique Canada - Apprentissage explicable](https://www.statcan.gc.ca/fr/science-donnees/reseau/apprentissage-explicable)

### Utilisation de Shapash


In [22]:
xpl = SmartExplainer(model=model)

In [23]:
xpl.compile(x=df_test.drop(columns="SalePrice", axis=1), y_target=df_test["SalePrice"])

INFO: Shap explainer type - <shap.explainers._tree.TreeExplainer object at 0x000002776F9FF890>


In [24]:
app = xpl.run_app(title_story="Houses Prices", port=8050)

INFO:root:Your Shapash application run on http://LAPTOP-245GVE4K:8050/
INFO:root:Use the method .kill() to down your app.


In [25]:
app.kill()

D'un point de vue global, voici une représentation des moyennes en valeurs absolues des valeurs de Shapley par variable :

<div style="text-align: center;"> <img src="../src/img/Feature importance MAC 1.png" alt="Feature importance MAC 1" /> </div>
Sans grande surprise, les variables ayant le plus grand impact sur le prix sont les surfaces et la qualité globale, ainsi que celles de la cuisine et de l'extérieur. D'un point de vue plus spécifique, il est tout à fait possible de détailler chaque variable et sa contribution à travers l'échantillon. J'ai choisi les quatre variables qui semblent contribuer le plus en valeur absolue sur le prix.

<div style="text-align: center;"> <img src="../src/img/GrLivArea Contrib 1.png" alt="GrLivArea Contrib 1" /> </div>
Plus la surface augmente, plus la contribution est positive. À noter que la relation ne semble pas vraiment linéaire.

<div style="text-align: center;"> <img src="../src/img/OverallQual Contrib 1.png" alt="OverallQual Contrib 1" /> </div>
Plus la qualité générale est élevée, plus la contribution est positive. Il semblerait intéressant de regrouper certaines modalités en raison de leur faible volume.

<div style="display: flex; justify-content: center; gap: 10px;"> <img src="../src/img/KitchenQual Contrib 1.png" alt="KitchenQual Contrib 1" style="max-width: 45%; height: auto;" /> <img src="../src/img/ExterQual Contrib 1.png" alt="ExterQual Contrib 1" style="max-width: 45%; height: auto;" /> </div>
Le même principe s'applique à la qualité de la cuisine et à celle de l'extérieur. D'un point de vue encore plus local, voici les valeurs pour la maison n°33 :

<div style="text-align: center;"> <img src="../src/img/id33 Contrib 1.png" alt="id33 Contrib 1" /> </div>
Ici, la surface contribue négativement au prix, contrairement à la qualité globale du bien et à la hauteur sous plafond du sous-sol. C'est ce qui est intéressant ici : pour cette maison en particulier, le sous-sol semble être un atout pour l'estimation.
